In [5]:
import numpy as np

In [6]:
def _partitionClusterIons(ions, coords, trapCapacity):
    partitions = [list(coords)]
    splitAxisIsX = True
    
    while max([len(p) for p in partitions])>trapCapacity:
        toSplit = [p for p in partitions if len(p)>trapCapacity]
        for p in toSplit:
            splitAxisVals = [float(c[int(splitAxisIsX)]) for c in p]
            medAxisVal = np.mean(splitAxisVals)
            p1 = []
            p2 = []
            for c, splAxisVal in zip(p, splitAxisVals):
                if splAxisVal <= medAxisVal:
                    p1.append(c)
                else:
                    p2.append(c)
            if p1:
                partitions.append(p1)
            if p2:
                partitions.append(p2)
        splitAxisIsX = not splitAxisIsX
        for p in toSplit:
            partitions.remove(p)

    coordsToIons = {(c[0], c[1]): i for c, i in zip(coords, ions)}
    clusters = []
    for p in partitions:
        clusterIons = [coordsToIons[(c[0], c[1])] for c in p]
        clusterCentre = np.mean(p, axis=0)
        clusters.append((clusterIons, clusterCentre))
    return clusters

ions = [0,1,2,3,4,5,6,7,8,9]
coords = [(0,0), (1,1), (2,0), (2,2), (3,1), (3,3), (4,0), (4,2), (5,1) ,(6,0)]
trapcap = 4
_partitionClusterIons(ions, coords, trapcap)

[([3, 5, 7], array([3.        , 2.33333333])),
 ([0, 1, 2, 4], array([1.5, 0.5])),
 ([6, 8, 9], array([5.        , 0.33333333]))]

In [7]:
class TriangleNode:
    def __init__(self, vertices, points, depth, capacity, collect=None):
        self.vertices = vertices
        self.points = points
        self.depth = depth
        self.capacity = capacity
        self.children: List[TriangleNode] = [] 
        self.clusters = [points]

        if collect is None:
            self.collect = []
        else:
            self.collect = collect
        
        if len(self.points) > self.capacity:
            self._subdivide()
        else:
            self.collect.append(self.points)
        

    def _subdivide(self):
        A, B, C = self.vertices
        mAB = (A + B)/2
        mBC = (B + C)/2
        mCA = (C + A)/2

        top = [A, mAB, mCA]
        left = [mAB, B, mBC]
        right = [mCA, mBC, C]
        center = [mAB, mBC, mCA]

        buckets = { 'top': [], 'left': [], 'right': [], 'center': [] }

        for p in self.points:
            wA, wB, wC = self._barycentric(p,A,B,C)

            if wA > 0.5:
                buckets['top'].append(p)
            elif wB > 0.5:
                buckets['left'].append(p)
            elif wC > 0.5:
                buckets['right'].append(p)
            else:
                buckets['center'].append(p)
        
        subtris = [top, left, right, center]
        for i in range(4):
            vert = subtris[i]
            pts = buckets[list(buckets.keys())[i]]
            if pts:
                childNode = TriangleNode(vert, pts, self.depth + 1, self.capacity, self.collect)
                self.children.append(childNode)

    def _barycentric(self, point, a,b,c):

        ba = b - a
        ca = c - a
        pa = point - a

        dot00 = np.dot(ba,ba)
        dot01 = np.dot(ba, ca)
        dot11 = np.dot(ca,ca)
        dot20 = np.dot(pa,ba)
        dot21 = np.dot(pa,ca)

        denom = dot00 * dot11 - dot01 * dot01

        if abs(denom) < 1e-10:
            return np.array([0,0,0]) 
        
        v = (dot11 * dot20 - dot01 * dot21) / denom
        w = (dot00 * dot21 - dot01 * dot20) / denom
        u = 1.0 - v - w

        return np.array([u,v,w])
        

In [8]:
c1 = np.array([np.array(p) for p in coords])
points = np.array([np.array(p) for p in [[3,3],[0,0],[6,0]]])
T = TriangleNode(points, c1, 0, 3)
T.collect

[[array([2, 2]), array([3, 3]), array([4, 2])],
 [array([0, 0]), array([1, 1]), array([2, 0])],
 [array([4, 0]), array([5, 1]), array([6, 0])],
 [array([3, 1])]]

In [9]:
from collections import deque
def clust(coords, trapCapacity):
    A = np.array([np.min(coords[:,0]) - 1.0, np.min(coords[:,1]) - 1.0])
    B = np.array([np.max(coords[:,0]) + 1.0, np.min(coords[:,1]) - 1.0])
    C = np.array([np.mean(coords[:,0]), np.max(coords[:,1]) + 1.0])
    all_clusters = []
    stack = deque([([A,B,C], coords, 0)])

    while stack:
        vertices, points, depth = stack.popleft()
        
        if len(points) <= trapCapacity:
            if len(points) > 0:
                all_clusters.append(points)
                continue
        
        A, B, C = vertices

        mAB = (A + B)/2
        mBC = (B + C)/2
        mCA = (C + A)/2

        subtriangles = {'top': [A, mAB, mCA],
                        'left': [mAB, B, mBC],
                        'right': [mCA, mBC, C],
                        'center': [mAB, mBC, mCA]}

        buckets = { 'top': [], 'left': [], 'right': [], 'center': [] }

        ba = B - A
        ca = C - A
        dot00 = np.dot(ba,ba)
        dot01 = np.dot(ba, ca)
        dot11 = np.dot(ca,ca)
        denom = dot00 * dot11 - dot01 * dot01

        #constant for div
        idenom = 1.0 / denom if abs(denom) > 1e-10 else 0.0

        for p in points:
            pa = p - A
            dot20 = np.dot(pa,ba)
            dot21 = np.dot(pa,ca)

            v = (dot11 * dot20 - dot01 * dot21) / denom
            w = (dot00 * dot21 - dot01 * dot20) / denom
            u = 1.0 - v - w

            if u > 0.5:
                buckets['top'].append(p)
            elif v > 0.5:
                buckets['left'].append(p)
            elif w > 0.5:
                buckets['right'].append(p)
            else:
                buckets['center'].append(p)
            
        # add to stack
        for i in ['top', 'left', 'right', 'center']:
            if buckets[i]:
                stack.append((subtriangles[i], np.array(buckets[i]), depth + 1))
    #print(all_clusters)
    return all_clusters

In [10]:
res = clust(c1, 3)
print(res)

[array([[0, 0],
       [1, 1],
       [2, 0]]), array([[4, 0],
       [5, 1],
       [6, 0]]), array([[2, 2],
       [3, 3],
       [4, 2]]), array([[3, 1]])]


In [34]:
import numpy as np
import numpy.typing as npt
from typing import (
    Sequence,
    Tuple,
)
from collections import deque
def TriangularPartitionIons(
    ions: list, coords: npt.NDArray[np.float64], trapCapacity: int):
    
    #ideally want to keep track of each triangle vertex regardless of which triangle it is in to make the maths easier
    # get vertices of outer triangle to use TriangleNode clustering
    A = np.array([np.min(coords[:,0]) - 1.0, np.min(coords[:,1]) - 1.0])
    B = np.array([np.max(coords[:,0]) + 1.0, np.min(coords[:,1]) - 1.0])
    C = np.array([np.mean(coords[:,0]), np.max(coords[:,1]) + 1.0])
    all_clusters = []
    stack = deque([(np.array([A,B,C]), coords, 0)])

    while stack:
        vertices, points, depth = stack.popleft()
        
        if len(points) <= trapCapacity:
            if len(points) > 0:
                all_clusters.append(points)
                continue
        
        A, B, C = vertices

        mAB = (A + B)/2
        mBC = (B + C)/2
        mCA = (C + A)/2

        subtriangles = {'top': [A, mAB, mCA],
                        'left': [mAB, B, mBC],
                        'right': [mCA, mBC, C],
                        'center': [mAB, mBC, mCA]}

        buckets = { 'top': [], 'left': [], 'right': [], 'center': [] }

        ba = B - A
        ca = C - A
        dot00 = np.dot(ba,ba)
        dot01 = np.dot(ba, ca)
        dot11 = np.dot(ca,ca)
        denom = dot00 * dot11 - dot01 * dot01

        #constant for div
        idenom = 1.0 / denom if abs(denom) > 1e-10 else 0.0

        for p in points:
            pa = p - A
            dot20 = np.dot(pa,ba)
            dot21 = np.dot(pa,ca)

            v = (dot11 * dot20 - dot01 * dot21) * idenom
            w = (dot00 * dot21 - dot01 * dot20) * idenom
            u = 1.0 - v - w

            if u > 0.5:
                buckets['top'].append(p)
            elif v > 0.5:
                buckets['left'].append(p)
            elif w > 0.5:
                buckets['right'].append(p)
            else:
                buckets['center'].append(p)
            
        # add to stack
        for i in ['top', 'left', 'right', 'center']:
            if buckets[i]:
                stack.append((subtriangles[i], np.array(buckets[i]), depth + 1))
    print(f'after triangular clusters: {all_clusters}')
    result = []
    for i in range(len(all_clusters)):
        clust = all_clusters[i]
        clusterCentre = np.mean(clust, axis=0)
        result.append((clust, clusterCentre))
    coordsToIons = {(c[0], c[1]): i for c, i in zip(coords, ions)}
    final_clusters = MergeUnderfilledClusters(result, trapCapacity, coordsToIons)
    print(f'final clusters: {final_clusters}')
    return result
    # change to reduce overclustering - add some check to see if clusters can be merged together if under capacity

def MergeUnderfilledClusters(
    clusters: list, trapCapacity: int, coordsToIons: dict):
    
    toCheck = []

    for i, (coords, center) in enumerate(clusters):
        toCheck.append({'size': len(coords), 'center': center, 'active': True, 'coords': coords})
    
    while True:
        
        candidates = [t for t in toCheck if t['active']]
        candidates.sort(key=lambda x: x['size'])
        merged = False
        #print(f'candidates = {candidates}')
        for i in range(len(candidates)):
            c1 = candidates[i]
            bestpartner = -1
            mindist = float('inf')

            for j in range(i+1, len(candidates)):
                c2 = candidates[j]
                dist = np.linalg.norm(c1['center'] - c2['center'])
                if dist < mindist and c1['size']+c2['size'] <= trapCapacity:
                    mindist = dist
                    bestpartner = j
            if bestpartner != -1:
                c2 = candidates[bestpartner]
                
                newsize = c1['size'] + c2['size']
                newcenter = (c1['size']*c1['center'] + c2['size']*c2['center']) / newsize

                newcluster = {'size': newsize, 'center': newcenter, 'active': True,
                              'coords': np.concatenate((c1['coords'], c2['coords']), axis=0)}
                
                c1['active'] = False
                c2['active'] = False
                toCheck.append(newcluster)

                merged = True
                break
        if not merged:
            break
    
    finalcoords = [t['coords'] for t in toCheck if t['active']]
    print(f'final coords: {finalcoords}')
    res = []
    for clust in finalcoords:
        ions = [coordsToIons[(c[0], c[1])] for c in clust] 
        res.append(ions)
    return res

In [35]:
TriangularPartitionIons([i for i in range(len(c1))], c1, 2) 

after triangular clusters: [array([[3, 1]]), array([[0, 0]]), array([[2, 0]]), array([[1, 1]]), array([[4, 0]]), array([[6, 0]]), array([[5, 1]]), array([[2, 2]]), array([[4, 2]]), array([[3, 3]])]
final coords: [array([[3, 1],
       [2, 0]]), array([[0, 0],
       [1, 1]]), array([[4, 0],
       [5, 1]]), array([[6, 0],
       [4, 2]]), array([[2, 2],
       [3, 3]])]
final clusters: [[4, 2], [0, 1], [6, 8], [9, 7], [3, 5]]


[(array([[3, 1]]), array([3., 1.])),
 (array([[0, 0]]), array([0., 0.])),
 (array([[2, 0]]), array([2., 0.])),
 (array([[1, 1]]), array([1., 1.])),
 (array([[4, 0]]), array([4., 0.])),
 (array([[6, 0]]), array([6., 0.])),
 (array([[5, 1]]), array([5., 1.])),
 (array([[2, 2]]), array([2., 2.])),
 (array([[4, 2]]), array([4., 2.])),
 (array([[3, 3]]), array([3., 3.]))]